In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# --- Fix output height for Jupyter notebooks ---
display(Markdown("<style>.output_scroll { max-height: none !important; }</style>"))

display(Markdown("### Graphical Calculation of Cross-Correlation Sequence"))
display(Markdown(r"Here, the graphical method for calculating the cross-correlation sequence $r_{xy}[\ell]$ is implemented step-by-step for the text signals: $x[n] = \{\dots, 0, 0, 2, -1, 3, +7, \mathbf{1}, +2, -3, 0, 0, \dots\}$ and $y[n] = \{\dots, 0, 0, 1, -1, 2, -2, \mathbf{4}, 1, -2, 5, 0, 0, \dots\}$."))

# Define relative time index support for x and y based on the text 
# Bold values correspond to n = 0.
# x[n]: values {2, -1, 3, 7, 1, 2, -3} at indices n = -4, -3, -2, -1, 0, 1, 2
n_x = np.array([-4, -3, -2, -1, 0, 1, 2])
x_vals = np.array([2.0, -1.0, 3.0, 7.0, 1.0, 2.0, -3.0])

# y[n]: values {1, -1, 2, -2, 4, 1, -2, 5} at indices n = -4, -3, -2, -1, 0, 1, 2, 3
n_y_orig = np.array([-4, -3, -2, -1, 0, 1, 2, 3])
y_vals = np.array([1.0, -1.0, 2.0, -2.0, 4.0, 1.0, -2.0, 5.0])

def update_correlation_calc(ell_val):
    # Common global display axis for n to show alignment clearly during shift
    n_min, n_max = -8, 8
    n = np.arange(n_min, n_max + 1)
    
    # Map x[n] onto the common n axis
    x_full = np.zeros_like(n, dtype=float)
    for idx, val in enumerate(n_x):
        mask = (n == val)
        x_full[mask] = x_vals[idx]
        
    # Shift y[n] by time lag ell -> y[n - ell] (Note: cross-correlation shifts y relative to x by ell)
    y_shifted_full = np.zeros_like(n, dtype=float)
    for idx, t_y in enumerate(n_y_orig):
        n_pos = t_y + ell_val
        mask = (n == n_pos)
        if np.any(mask):
            y_shifted_full[mask] = y_vals[idx]
            
    # Sample-by-sample multiplication v_ell[n] = x[n] * y[n - ell]
    v_ell = x_full * y_shifted_full
    
    # Summation to find r_xy[ell]
    r_xy_current = np.sum(v_ell)
    
    # Plotting
    fig, axes = plt.subplots(3, 1, figsize=(10, 7.5), sharex=True)
    
    # Plot x[n]
    axes[0].stem(n, x_full, linefmt='b-', markerfmt='bo', basefmt='k-')
    axes[0].set_title(r'Reference Signal $x[n]$', fontsize=9.5, fontweight='bold', color='darkblue')
    axes[0].set_ylabel('Amplitude', fontsize=8.5)
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].set_ylim(-4.5, 8.5)
    
    # Plot shifted signal y[n - ell]
    axes[1].stem(n, y_shifted_full, linefmt='g-', markerfmt='go', basefmt='k-')
    axes[1].set_title(r'Shifted Signal $y[n-\ell]$ for lag $\ell = %d$' % ell_val, fontsize=9.5, fontweight='bold', color='darkgreen')
    axes[1].set_ylabel('Amplitude', fontsize=8.5)
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].set_ylim(-3.5, 6.0)
    
    # Plot product sequence v_ell[n]
    axes[2].stem(n, v_ell, linefmt='r-', markerfmt='ro', basefmt='k-')
    axes[2].set_title(r'Product Sequence $\nu_{\ell}[n] = x[n]y[n-\ell]$  $\rightarrow$  Sum ($r_{xy}[%d]$) = %.1f' % (ell_val, r_xy_current), fontsize=9.5, fontweight='bold', color='darkred')
    axes[2].set_xlabel('Index $n$', fontsize=8.5)
    axes[2].set_ylabel('Amplitude', fontsize=8.5)
    axes[2].grid(True, linestyle='--', alpha=0.6)
    axes[2].set_ylim(-16.0, 40.0)
    
    plt.tight_layout()
    plt.show()

interactive_corr = widgets.interactive(
    update_correlation_calc,
    ell_val=widgets.IntSlider(value=0, min=-7, max=7, step=1, description='Time Lag (\u2113):', style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'))
)
display(interactive_corr)